# 08 Export Decoder And Cpu Benchmark

Purpose: execute the 08 export decoder and cpu benchmark stage.
Required inputs: DATA_ROOT plus artifacts from prior stages when listed.
Expected outputs: decoder_state_dict.pt and resource_metrics.json.
Fold: FOLD; seed: SEED; all unexecuted results remain `not_run`.


In [1]:
from pathlib import Path
import os, sys, platform, torch

print("=== CHECKING /kaggle/input ===")
input_dir = Path("/kaggle/input")
if input_dir.exists():
    items = list(input_dir.iterdir())
    print(f"Contents of /kaggle/input ({len(items)} items): {[p.name for p in items]}")
    for it in items:
        if it.is_dir():
            print(f"  Folder {it.name} contains {len(list(it.iterdir()))} items")
else:
    print("/kaggle/input does not exist!")

# Robust discovery of DATA_ROOT
DATA_ROOT = None
if os.environ.get('PPGDALIA_ROOT'):
    p = Path(os.environ['PPGDALIA_ROOT'])
    if p.exists():
        DATA_ROOT = p

if DATA_ROOT is None and input_dir.exists():
    for p in input_dir.rglob("*.pkl"):
        if any(f"S{i}" in p.name.upper() for i in range(1, 16)):
            rel = p.relative_to(input_dir)
            DATA_ROOT = input_dir / rel.parts[0]
            print(f"Auto-discovered DATA_ROOT via subject files: {DATA_ROOT}")
            break

if DATA_ROOT is None:
    _candidates = [
        Path('/kaggle/input/ppg-dalia-dataset'),
        Path('/kaggle/input/ppg-dalia'),
        Path('/kaggle/input/datasets/ameersifat53/ppg-dalia-dataset')
    ]
    DATA_ROOT = next((p for p in _candidates if p.exists() and p.is_dir()), _candidates[0])

OUTPUT_ROOT = Path('/kaggle/working/ppg-cvae-artifacts')
FOLD = 1
SEED = 42
print({'DATA_ROOT': str(DATA_ROOT), 'OUTPUT_ROOT': str(OUTPUT_ROOT), 'device': 'cuda' if torch.cuda.is_available() else 'cpu', 'python': sys.version, 'torch': torch.__version__})
if not DATA_ROOT.exists():
    raise FileNotFoundError(f'Configure DATA_ROOT to the mounted PPG-DaLiA dataset: {DATA_ROOT}')


=== CHECKING /kaggle/input ===
Contents of /kaggle/input (1 items): ['datasets']
  Folder datasets contains 1 items
Auto-discovered DATA_ROOT via subject files: /kaggle/input/datasets
{'DATA_ROOT': '/kaggle/input/datasets', 'OUTPUT_ROOT': '/kaggle/working/ppg-cvae-artifacts', 'device': 'cpu', 'python': '3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]', 'torch': '2.11.0+cpu'}


In [2]:
# Locate this repository without assuming a Kaggle dataset slug.
repo_candidates = [Path.cwd(), Path.cwd() / 'Final_IOT_for_AI']
repo_root = next((p for p in repo_candidates if (p / 'src').exists()), None)
if repo_root is None:
    raise FileNotFoundError('Could not locate repository src/ directory; upload/clone the repository first.')
sys.path.insert(0, str(repo_root / 'src'))
from ppg_cvae.data.splits import get_fold
print(get_fold(FOLD))


FoldSubjects(fold_id=1, train_subjects=frozenset({'S14', 'S8', 'S10', 'S11', 'S15', 'S13', 'S9', 'S7', 'S12'}), val_subjects=frozenset({'S5', 'S6', 'S4'}), test_subjects=frozenset({'S2', 'S3', 'S1'}))


In [3]:
from ppg_cvae.models.cvae import ConditionalVAE
from ppg_cvae.evaluation.resources import measure_resources
model=ConditionalVAE()
print(measure_resources(model, iterations=1000, warmup=100))


{'parameter_counts': {'encoder': 282544, 'decoder': 84081, 'cvae': 366625}, 'decoder_latency_ms_median': 0.2789030000087678, 'decoder_latency_ms_p95': 0.6975583999690116}
